# Phase 2 DP Retirement — Detailed Report

**CME 241: Reinforcement Learning for Finance — Winter 2026**  
**Stanford University**

---

## Abstract

This report documents the **Phase 2** retirement asset-allocation model: a finite-horizon MDP solved by exact **Dynamic Programming (backward induction)**. The model considers three strategies (VFSTX, VBMFX, VFINX), age-dependent CRRA risk aversion, and terminal utility at retirement. All parameters, data sources, equations, and replication steps are specified so that results can be reproduced exactly.

## Table of Contents

1. [Replication Instructions](#1-replication-instructions)
2. [Data Sources and Parameter Assumptions](#2-data-sources-and-parameter-assumptions)
3. [Model Formulation (MDP)](#3-model-formulation-mdp)
4. [Utility and Transition](#4-utility-and-transition)
5. [DP Solution (Backward Induction)](#5-dp-solution-backward-induction)
6. [Simulation and Policies](#6-simulation-and-policies)
7. [Results and Figures](#7-results-and-figures)
8. [Interpretation and Limitations](#8-interpretation-and-limitations)

---
## 1. Replication Instructions

To replicate all results and figures:

1. **Environment**: Python 3.8+ with `numpy`, `pandas`, `matplotlib`, `yfinance`. Set `np.random.seed(42)` for reproducibility.

2. **Run the main notebook**: Execute `phase2_dp_retirement.ipynb` from top to bottom. This will:
   - Download historical returns (VFINX, VBMFX, VFSTX) via yfinance for 1990–2024.
   - Build the config, wealth grid, and empirically calibrated return scenarios.
   - Run backward induction to obtain value function `V` and policy `π*`.
   - Run 10,000 Monte Carlo paths per policy and save figures to the project directory.

3. **Outputs**: The main notebook writes `fig1_policy_and_value.png` through `fig5_mean_utility_comparison.png`. This report loads and displays those figures in Section 7.

4. **Key seeds and sizes**: `np.random.seed(42)`; simulation uses `n_paths=10_000` and default RNG seed 0 inside `simulate_paths`.

---
## 2. Data Sources and Parameter Assumptions

### 2.1 Config (exact values for replication)

| Parameter | Value | Description |
|-----------|-------|-------------|
| `start_age` | 30 | First decision age |
| `retirement_age` | 65 | Terminal age |
| `initial_wealth` | 100,000 | Starting wealth ($) |
| `annual_contribution` | 10,000 | Fixed annual contribution ($) |
| `w_min` | 1,000 | Minimum wealth (grid floor) ($) |
| `w_max` | 5,000,000 | Maximum wealth (grid cap) ($) |
| `n_bins` | 60 | Log-spaced wealth grid size |
| `gamma_0` | 1.0 | CRRA γ at age 30 |
| `gamma_slope` | 0.03 | Increase in γ per year |
| `w_ref` | 1,000,000 | CRRA normalization ($); U(w_ref)=0 |
| `discount` | 0.99 | Per-step discount factor |
| `intermediate_alpha` | 0.01 | Weight on dense per-step log-wealth reward |

### 2.2 Hand‑calibrated return scenarios (before empirical fit)

- **VFSTX**: (1%, 0.28), (3%, 0.32), (4.5%, 0.40) — bear / neutral / bull.
- **VBMFX**: (10%, 0.28), (4%, 0.32), (2%, 0.40) — initial hand-calibrated values; overwritten by empirical fit.
- **VFINX**: (-15%, 0.28), (10%, 0.32), (27%, 0.40).

**Six actions** are available: three pure picks (VFSTX=0, VBMFX=1, VFINX=2) and three equal-weight 50/50 blends (VFSTX+VBMFX=3, VFSTX+VFINX=4, VBMFX+VFINX=5).

After the **empirical calibration** step, these are replaced by 3‑scenario fits to historical annual returns (VFINX, VBMFX, VFSTX) over 1990–2024, preserving approximate probability splits (≈28% / 32% / 40%). Data source: **yfinance** (Yahoo Finance), adjusted close, year‑end, 35 years aligned.

**Empirical calibration procedure**: Sort each asset’s annual returns; split into three buckets with counts proportional to p_bear=0.28, p_neutral=0.32, remainder=bull. Mean return in each bucket and empirical frequency give the three (return, probability) pairs that overwrite `cfg.return_scenarios` for that asset.

### 2.3 Strategy names and indices

- `VFSTX = 0`, `VBMFX = 1`, `VFINX = 2`.
- Names: `["VFSTX", "VBMFX", "VFINX"]`.

---
## 3. Model Formulation (MDP)

- **State**: (t, w_i) — time step t (age 30+t) and wealth bin index i.
- **Action**: a ∈ {0,…,5} — six actions: VFSTX (0), VBMFX (1), VFINX (2), 50/50 VFSTX+VBMFX (3), 50/50 VFSTX+VFINX (4), 50/50 VBMFX+VFINX (5).
- **Transition**: W_{t+1} = (W_t + c)×(1 + r_t), then clip to w_min and snap to nearest log-spaced wealth bin.
- **Reward**: Dense intermediate reward r_t = 0.01·(log W_{t+1} − log W_t) for t < T; at retirement (t=T): r_T = U(W_T; γ(65)) with CRRA utility.
- **Horizon**: T = 35 steps (ages 30–64; terminal at 65).
- **Risk aversion**: γ(age) = γ_0 + gamma_slope×(age−30).
- **Switching cost**: δ_t = 0.005·W_t when action changes from t−1 to t.

**Bellman** (discount β = 0.99):  
- Terminal: V(T, w_i) = U(w_i; γ(65)).  
- Recursion: V(t, w_i) = max_a [ r_t + β · Σ_j P(w_j | w_i, a) V(t+1, w_j) ].  
- Policy: π*(t, w_i) = argmax_a [ r_t + β · Σ_j P(w_j | w_i, a) V(t+1, w_j) ].

---
## 4. Utility and Transition

**CRRA (normalized by w_ref):**
- U(W; γ) = ((W/w_ref)^(1−γ) − 1) / (1−γ) for γ≠1; ln(W/w_ref) for γ=1.  
- W clipped to ≥ 1e-6 before normalization.

**Wealth grid**: `wealth_grid = np.logspace(log10(w_min), log10(w_max), n_bins)`.  
**Binning**: nearest bin in log10(wealth) space; functions `nearest_bin_idx(w)` and `nearest_bin_idx_vec(w_arr)`.

**Transition**: For each (wealth_bin_idx, action), loop over return scenarios (r, p), compute next wealth = (w + c)*(1+r), clip to w_min, then `next_bin = nearest_bin_idx(next_wealth)`. Return list of (next_bin, probability).

---
## 5. DP Solution (Backward Induction)

1. Allocate V[T_steps+1, N] and pi[T_steps, N].
2. Terminal: V[T_steps, i] = crra_utility(wealth_grid[i], gamma_from_age(65)) for all i.
3. For t = T_steps−1 down to 0, for each wealth bin i: compute expected value for each action a using get_transitions(i, a), take max over a → store in V[t,i] and pi[t,i].

Result: V shape (36, 60), pi shape (35, 60).

---
## 6. Simulation and Policies

**Forward simulator**: For each path, start at initial_wealth; at each step get action from policy(age, wealth_bin), sample return from cfg.return_scenarios[action], update wealth = (wealth + annual_contribution)*(1+return), clip to w_min. Repeat for T_steps. Return terminal wealth array and full path matrix.

**Policies evaluated** (each returns action index 0–5):
- **DP Optimal**: π*(age, wealth_bin) from backward induction.
- **Always VFSTX / Always VBMFX / Always VFINX**: constant pure-pick action.
- **50/50 VFSTX+VBMFX / 50/50 VFSTX+VFINX / 50/50 VBMFX+VFINX**: constant blend action.
- **Glide Path**: VFINX for ages 30–49, VBMFX for 50–59, VFSTX for 60–64.

**Simulation settings**: n_paths = 10,000; seed = 0 inside simulate_paths; np.random.seed(42) in main notebook for any other randomness.

---
## 7. Results and Figures

The main notebook saves five figures. Below we load and display them (run `phase2_dp_retirement.ipynb` first so the PNG files exist).

In [ ]:
import os
from IPython.display import Image, display
fname = "fig1_policy_and_value.png"
if os.path.isfile(fname):
    display(Image(filename=fname, width=820))
else:
    print(f"[Not found: {fname}] Run phase2_dp_retirement.ipynb to generate figures.")

#### Analysis — Figure 1: Optimal Policy and Value Function

**Policy heatmap (left panel).** The DP policy π\*(age, wealth\_bin) exhibits three wealth-driven bands that are nearly time-invariant across the 35-year horizon:

- **Low wealth (log₁₀W ≲ 5.0, roughly W ≲ $100k):** VFINX (equity) is chosen at virtually every age. At low wealth the investor is far below the $1 M reference; the CRRA utility gradient is steep here, so the expected excess return of equity dominates its variance.
- **Mid wealth (~$100 k–$600 k):** VFSTX or VBMFX appears. This is the region where normalized CRRA utility U(W/w\_ref; γ) is most concave: a large equity drawdown here would drop the investor back into deeply negative-utility territory, and the DP correctly trades expected growth for variance protection.
- **High wealth (log₁₀W ≳ 5.7, roughly W ≳ $500 k):** VFINX re-emerges. Once wealth comfortably exceeds w\_ref, the utility function flattens and equity exposure again adds expected value with manageable marginal utility loss.

The near-vertical stripe pattern (policy barely changing with age) reflects the discrete-scenario return model: transition probabilities are age-independent, so only the terminal boundary γ(age) creates age sensitivity — which is modest at 60-bin resolution.

**Value function heatmap (right panel).** V(age, wealth\_bin) increases monotonically with wealth (brighter = higher expected utility) and decreases with age (darker near retirement given the same wealth). The gradient is steepest near w\_ref at all ages, confirming that the optimization is most sensitive to wealth in the vicinity of the $1 M target. Smooth, log-shaped contours are consistent with the power-law structure of CRRA utility on the log-spaced wealth grid.

In [ ]:
import os
from IPython.display import Image, display
fname = "fig2_strategy_by_age.png"
if os.path.isfile(fname):
    display(Image(filename=fname, width=820))
else:
    print(f"[Not found: {fname}] Run phase2_dp_retirement.ipynb to generate figures.")

#### Analysis — Figure 2: Strategy Frequency by Age

This stacked bar chart aggregates the optimal policy across all 60 wealth bins, showing what fraction of bins chooses each strategy at each calendar year (= age + birth year 1960):

- **VFINX (red) dominates throughout**, reflecting that equity is optimal at both the lowest and highest wealth bins, which together account for the majority of the grid. Only the mid-wealth bins near w\_ref prefer conservative strategies.
- **VFSTX and VBMFX (green and blue)** occupy noticeable fractions in mid-career years. These fractions shift slightly with age because γ(age) = 2.0 + 0.03×(age−30) rises at 0.03 per year, marginally lowering the wealth threshold at which conservative strategies become preferable.
- **The pattern is not a monotone glide path.** A conventional target-date fund would shift equity → bonds with age alone. Here the DP shows a roughly flat profile: the conservative zone is defined by *proximity to w\_ref*, not age. An investor who accumulates quickly passes through and beyond this zone early; a slower accumulator may not reach it until age 55.
- **Implication for communication:** Marginalizing over wealth obscures the two-dimensional structure revealed in Figure 1. The chart is useful for confirming that equity dominates, but the per-wealth-bin policy (Figure 1) is the more actionable result.

In [ ]:
import os
from IPython.display import Image, display
fname = "fig3_wealth_paths_dp.png"
if os.path.isfile(fname):
    display(Image(filename=fname, width=820))
else:
    print(f"[Not found: {fname}] Run phase2_dp_retirement.ipynb to generate figures.")

#### Analysis — Figure 3: Sample Wealth Paths (DP Optimal Policy)

The 200 Monte Carlo trajectories under the DP policy illustrate the *distribution* of outcomes and how the policy shapes the path:

- **Fan-out:** Paths diverge substantially from the common starting point ($100 k at age 30). By mid-career (age 45–50) the interquartile range spans roughly 2–3× in wealth. This spread is driven by VFINX's high return variance (bull +27 %, neutral +10 %, bear −15 %) during the low-wealth growth phase.
- **Regime-driven plateaus:** Paths that experience back-to-back bear scenarios early in life plateau at lower wealth levels and the DP policy switches them to VFSTX or VBMFX — visible as flat, low-volatility segments in the middle years. Paths hitting bull streaks grow rapidly past w\_ref and return to equity.
- **Divergence near retirement:** Even though all paths start identically, the terminal wealth distribution at age 65 remains wide. This irreducible variance reflects the stochastic nature of returns; no deterministic rule eliminates it.
- **Median outcome:** The typical (median) path crosses $500 k–$1 M by retirement, consistent with the terminal wealth statistics in Section 7.
- **Downside paths:** The worst-case trajectories (repeated equity bear years) end near $200 k–$400 k, illustrating the left-tail risk of an equity-heavy policy even when it is CRRA-optimal in expectation.

In [ ]:
import os
from IPython.display import Image, display
fname = "fig4_terminal_wealth_histograms.png"
if os.path.isfile(fname):
    display(Image(filename=fname, width=820))
else:
    print(f"[Not found: {fname}] Run phase2_dp_retirement.ipynb to generate figures.")

#### Analysis — Figure 4: Terminal Wealth Histograms by Policy

The histograms reveal fundamental differences in the *shape* of the terminal-wealth distribution across policies:

- **Always VFSTX:** Narrowest distribution, concentrated near $350 k–$500 k. The three return scenarios (1 %, 3 %, 4.5 %) map to very similar 35-year compounding outcomes — almost no probability of meaningful accumulation, and no left tail worth mentioning.
- **Always VBMFX:** Slightly wider and higher than VFSTX. VBMFX's asymmetric returns (highest in bear markets via flight-to-safety) add moderate spread; its calibrated expected return exceeds VFSTX's.
- **Always VFINX:** Broad, right-skewed with a long tail reaching $3–5 M. The equity geometric mean dominates over 35 years, but the distribution also has meaningful probability below $500 k from sustained bad-return sequences.
- **Glide Path:** Intermediate spread, roughly symmetric around $800 k–$1.2 M. The forced de-risking after age 44 truncates the left tail (fewer bear catastrophes) at the cost of also truncating the right tail (less equity compounding).
- **DP Optimal:** Broadly similar to Always VFINX for good outcomes, with a somewhat better-supported lower tail due to mid-career switches to VFSTX/VBMFX near w\_ref.
- **Key takeaway:** CRRA utility up-weights left-tail protection relative to right-tail upside. The DP's shape (even if similar in median to Always VFINX) can yield different mean utility precisely because it manages the left tail more carefully.

In [ ]:
import os
from IPython.display import Image, display
fname = "fig5_mean_utility_comparison.png"
if os.path.isfile(fname):
    display(Image(filename=fname, width=820))
else:
    print(f"[Not found: {fname}] Run phase2_dp_retirement.ipynb to generate figures.")

#### Analysis — Figure 5: Mean Terminal CRRA Utility by Policy

The bar chart is the definitive ranking under the objective U(W\_T; γ(65), w\_ref = $1 M):

- **Always VFINX achieves the highest mean utility (0.4632).** Over 35 years the equity risk premium dominates: with γ(65) ≈ 2.75–3.05 and a $1 M target, the expected log-utility gain from equities outweighs the utility cost of variance.
- **DP Optimal ranks second (0.4552)**, confirming that the backward-induction policy correctly identifies equity-heavy allocations as optimal, with modest diversification into the 50/50 VFSTX+VFINX blend at mid-wealth bins.
- **Glide Path ranks third (0.4168)**, showing that forced de-risking by age alone sacrifices equity compounding in early career without fully capturing the DP's wealth-contingent logic.
- **Always VBMFX (0.2637)** reflects moderate returns with some volatility — better than VFSTX but far below equity-heavy strategies.
- **Always VFSTX has the lowest mean utility (0.2100):** terminal wealth near $1.35 M on a $1 M reference places outcomes in the shallow positive CRRA region.
- **Design implication:** Mean utility comparisons are sensitive to the choice of w\_ref and γ. The qualitative ordering (equity > blend > conservative) is robust. The DP's slight underperformance vs Always VFINX is driven by wealth-grid discretisation at 60 bins; a finer grid or continuous-state DQN closes this gap.

---
## 8. Interpretation and Limitations

### 8.1 Policy ranking (mean terminal utility)

Ordering from highest to lowest: **Always VFINX (0.4632) > DP Optimal (0.4552) > Glide Path (0.4168) > Always VBMFX (0.2637) > Always VFSTX (0.2100)**. The DP ranks second — its backward-induction policy correctly concentrates on VFINX while diversifying into the 50/50 VFSTX+VFINX blend at mid-wealth bins. Always VFINX marginally exceeds the DP because the equity risk premium over 35 years is strong enough that the occasional blend diversification costs more than it saves at this grid resolution. Finer grids (e.g. 100–200 bins) or a continuous-state DQN further close the gap.

### 8.2 Policy structure (Figure 1)

- **Low wealth**: DP chooses VFINX (need growth).
- **Mid wealth (~$100k–$500k)**: VFSTX or VBMFX (CRRA curvature near w_ref).
- **High wealth**: VFINX again (already near/above target).

### 8.3 Limitations

- Three discrete return scenarios per asset; no full fat tails or serial correlation.
- Fixed contribution; no income or inflation.
- 60 log-spaced bins can make VFSTX appear quasi-deterministic (tightly clustered returns).
- i.i.d. returns; no regime persistence.
- No transaction costs or intermediate consumption.

### 8.4 Replication checklist

- Python 3.8+, numpy, pandas, matplotlib, yfinance.
- `np.random.seed(42)` at start.
- Run `phase2_dp_retirement.ipynb` top to bottom (downloads data, calibrates, solves DP, runs 10k paths per policy, saves fig1–fig5).
- This report notebook: run after the main notebook to view figures and refer to this documentation.

In [ ]:
# Optional: run the main notebook to regenerate all results and figures.
# Uncomment and run this cell if you want this report to replicate everything in one go.
# (Requires nbformat and nbconvert: pip install nbformat nbconvert)

# import subprocess, sys
# subprocess.run([sys.executable, "-m", "jupyter", "nbconvert", "--to", "notebook", "--execute",
#                "phase2_dp_retirement.ipynb", "--output", "phase2_dp_retirement_executed.ipynb"], check=True)
# Then re-run the figure cells above to display the newly generated figures.